# 07. CPU and GPU Heterogeneous Scheduling | CPU 与 GPU 异构调度

**难度：** Medium | **环境：** GPU optional | **标签：** `系统工程`, `CPU/GPU`, `异构调度` | **目标人群：** 系统性能入门者

![CPU 与 GPU 的阶段放置和重叠](../docs/public/01_Hardware_Math_and_Systems/07_heterogeneous_stage_overlap.svg)

---

## 本节导读

大模型系统里的很多等待，并不是 GPU 算得不够快，而是任务放错了地方：控制流留在 CPU 更合适，还是应该尽早下放到 GPU；数据该不该提前搬过去；通信和计算能不能并起来做。这些判断一旦错位，GPU 再强也会被 host-device 往返和同步点拖住。

这一页在整个教程的纵向主线里属于 `Part 01` 的系统执行基础页，优先服务 `推理优化路线` 和后续的 kernel 下沉主线。学完这里，后面再看 `08 / 15 / 17 / 29` 以及 `66 / 67` 时，你会更容易把“任务放置 -> 数据搬运 -> 调度重叠”连成一条判断链；如果这里没学明白，后面常见的问题是看得懂异步、stream 和 overlap 这些词，却判断不清等待究竟来自 host-device 往返、同步点，还是任务本来就放错了位置。按专题归类，这一页主要属于 `推理优化路线` 的系统前置，也和 `Profiling 专题`、`编译与图优化专题` 共享执行视角。

**关键词：** `host`, `device`, `PCIe`

---

## 前置阅读
**导语：** 这一页先把 CPU / GPU 协同、通信延迟和调度重叠讲清楚，再决定什么时候该把任务留在 CPU，什么时候该把任务交给 GPU。
- [Group 1C: Communication, Parallelism and Fault Tolerance | 1C: 通信、并行与容错](./1C.md)
- [15. CUDA Execution Model | CUDA 执行模型](./15_CUDA_Execution_Model.ipynb)
- [17. CUDA Stream and Asynchrony | CUDA Stream 与异步执行](./17_CUDA_Stream_and_Asynchrony.ipynb)

## Q1：Host 和 Device 分别扮演什么角色？


Host 通常负责控制流、数据准备和调度，Device 负责大规模并行计算。

两者之间的核心瓶颈，往往不是“谁更强”，而是数据在两边之间搬运的成本和频率。

所以异构调度的第一件事，是先知道哪个环节适合留在 CPU，哪个环节应该下放到 GPU。


### Q1 机制计算器：比较阶段放在 CPU 或 GPU 的端到端时间

固定同一阶段的输入输出字节数，比较 CPU 时间与“GPU 计算 + 双向搬运”时间。GPU 单独计算更快，不代表端到端一定更快。

In [ ]:
def stage_placement(cpu_ms, gpu_ms, input_bytes, output_bytes, bandwidth_gbps, fixed_transfer_ms=0.0):
    """比较阶段留在 CPU 与迁移到 GPU 的串行端到端时间。"""
    if min(cpu_ms, gpu_ms, input_bytes, output_bytes, fixed_transfer_ms) < 0:
        raise ValueError('times and bytes must be non-negative')
    if bandwidth_gbps <= 0:
        raise ValueError('bandwidth_gbps must be positive')
    transfer_ms = (input_bytes + output_bytes) * 8 / (bandwidth_gbps * 1e9) * 1e3 + fixed_transfer_ms
    gpu_path_ms = gpu_ms + transfer_ms
    return {'cpu_path_ms': cpu_ms, 'gpu_compute_ms': gpu_ms, 'transfer_ms': transfer_ms, 'gpu_path_ms': gpu_path_ms, 'placement': 'gpu' if gpu_path_ms < cpu_ms else 'cpu'}

small_stage = stage_placement(2.0, 0.4, 64e6, 64e6, 16.0, fixed_transfer_ms=0.1)
large_stage = stage_placement(100.0, 8.0, 64e6, 64e6, 16.0, fixed_transfer_ms=0.1)
assert small_stage['placement'] == 'cpu'
assert large_stage['placement'] == 'gpu'
print('small stage:', small_stage)
print('large stage:', large_stage)

## Q2：CUDA Streams 是如何隐藏通信延迟的？


CUDA Streams 的作用，是让数据搬运和计算有机会重叠。

如果把拷贝和计算放在合适的队列里，GPU 就不必一边等数据一边空转，而是可以在一部分数据搬运时去做另一部分计算。

这不是让通信消失，而是把通信藏进了别的工作里。


### Q2 对照验证：重叠能隐藏多少搬运时间

固定搬运与计算时间，对比串行时间和理想重叠时间；隐藏量不会超过两者中的较小值。

In [ ]:
def overlap_timeline(copy_ms, compute_ms, dependency_ms=0.0):
    """计算串行时间、受依赖约束的重叠时间和被隐藏的搬运时间。"""
    if min(copy_ms, compute_ms, dependency_ms) < 0 or dependency_ms > copy_ms:
        raise ValueError('invalid timeline')
    serial_ms = copy_ms + compute_ms
    overlappable_copy_ms = copy_ms - dependency_ms
    elapsed_ms = dependency_ms + max(overlappable_copy_ms, compute_ms)
    hidden_ms = serial_ms - elapsed_ms
    return {'serial_ms': serial_ms, 'elapsed_ms': elapsed_ms, 'hidden_ms': hidden_ms}

full_overlap = overlap_timeline(8.0, 20.0)
dependency_limited = overlap_timeline(8.0, 20.0, dependency_ms=5.0)
assert full_overlap['hidden_ms'] == 8.0
assert 0 <= dependency_limited['hidden_ms'] <= 8.0
print(full_overlap)
print(dependency_limited)

## Q3：CPU Offload 在训练和推理中分别怎么用？


CPU Offload 的目标，是把一部分暂时不需要常驻 GPU 的状态挪到 CPU，缓解显存压力。

在训练中，常见对象可能是优化器状态、部分参数或激活；在推理中，则可能是权重分层驻留或临时状态管理。

它的核心代价是更频繁的数据搬运，所以通常要和调度策略一起看，而不能只看“显存省了多少”。


### Q3 机制计算器：把容量收益与搬运时间分开记录

容量收益使用 GB，搬运代价使用 ms/step，二者不能直接相减。由状态大小、带宽和每步往返次数计算暴露搬运时间。

In [ ]:
def offload_ledger(state_gb, bandwidth_gbps, round_trips_per_step=1, overlap_ratio=0.0):
    """记录卸载容量收益、总搬运时间和未被重叠隐藏的时间。"""
    if state_gb < 0 or bandwidth_gbps <= 0 or round_trips_per_step < 0:
        raise ValueError('invalid offload configuration')
    if not 0 <= overlap_ratio <= 1:
        raise ValueError('overlap_ratio must be in [0, 1]')
    transfer_gb = state_gb * 2 * round_trips_per_step
    transfer_ms = transfer_gb * 8 / bandwidth_gbps * 1e3
    exposed_ms = transfer_ms * (1 - overlap_ratio)
    return {'saved_vram_gb': state_gb, 'transfer_gb_per_step': transfer_gb, 'transfer_ms': transfer_ms, 'exposed_ms': exposed_ms}

ledger = offload_ledger(4.0, bandwidth_gbps=256.0, round_trips_per_step=1, overlap_ratio=0.75)
assert ledger['saved_vram_gb'] == 4.0
assert ledger['exposed_ms'] <= ledger['transfer_ms']
print(ledger)

## Q4：怎么判断一套异构调度方案是不是“看起来很并行，但实际没省多少时间”？


判断标准不是“有没有用了很多并行词汇”，而是实际是否把等待时间压下去了。

如果 CPU 和 GPU 之间仍然频繁同步、通信和计算没有重叠、或者某个阶段仍然在卡住整条流水线，那么表面并行并不会带来实际收益。

所以异构调度最终还是要回到吞吐、延迟和重叠效率上看。


### Q4 判定：时间线是否产生了可观测收益

使用串行基线、实际时间、GPU 有效工作和同步等待做硬判断，不用任务数量代替收益证据。

In [ ]:
def parallelism_evidence(serial_ms, elapsed_ms, useful_gpu_ms, sync_wait_ms, min_speedup=1.05, max_sync_fraction=0.1):
    """根据可观测时间判断并行方案是否真正缩短关键路径。"""
    if min(serial_ms, elapsed_ms, useful_gpu_ms, sync_wait_ms) < 0 or elapsed_ms == 0:
        raise ValueError('invalid timeline')
    speedup = serial_ms / elapsed_ms
    gpu_utilization = min(useful_gpu_ms / elapsed_ms, 1.0)
    sync_fraction = sync_wait_ms / elapsed_ms
    accepted = speedup >= min_speedup and sync_fraction <= max_sync_fraction
    return {'speedup': speedup, 'gpu_utilization': gpu_utilization, 'sync_fraction': sync_fraction, 'accepted': accepted}

good = parallelism_evidence(30.0, 20.0, 18.0, 1.0)
fake = parallelism_evidence(30.0, 29.0, 18.0, 8.0)
assert good['accepted'] is True
assert fake['accepted'] is False
print('effective:', good)
print('fake:', fake)

## ⚠️ 常见误区

- 并行任务多不等于更快。
- 异构调度的关键是重叠和边界划分。
- Offload 省显存，但会带来搬运成本。
- 先看吞吐和延迟，再看并行标签。

## 相关阅读
**导语：** 如果想继续把异构调度和 launch / graph 的关系补完整，可以接着看这些页。
- [Part 03: Triton Kernel Development | 第三部分：Triton 算子开发](../03_Triton_Kernels/intro.md)
- [05. Triton 性能调优与基准测试 (Autotune & Profiling)](../03_Triton_Kernels/05_Triton_Autotune_and_Profiling.ipynb)
- [18. CUDA Graph and JIT Compile | CUDA Graph 与 JIT 编译](../04_CUDA_and_System_Optimization/18_CUDA_Graph_and_JIT_Compile.ipynb)
---